# 운수종사자 인지 검사 심층 분석 (Advanced EDA)
---
검사 항목별 의미를 이해하고 인지적 특성과 사고 위험의 관계를 분석합니다.

## 0. 환경 설정 및 데이터 로드

In [23]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  # WSL 환경에서 안전한 백엔드 사용
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# 시각화 설정
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10

# seaborn 스타일 설정을 안전하게 처리
try:
    sns.set_style('whitegrid')
    sns.set_palette('husl')
except Exception as e:
    print(f"Seaborn 스타일 설정 건너뜀: {e}")

print("라이브러리 로드 완료!")

라이브러리 로드 완료!


In [24]:
# 데이터 로드 (메모리 최적화)
print("데이터 로드 중...")
train_meta = pd.read_csv("./data/train.csv")

# 먼저 작은 샘플링을 한 후 병합 (메모리 절약)
SAMPLE_SIZE = 50000  # 메모리와 통계적 유의성 균형

# A 검사: 샘플링 후 로드
train_A = pd.read_csv("./data/train/A.csv")
print(f"A 검사 원본: {train_A.shape}")

# 샘플링 후 병합
sampled_ids = train_A['Test_id'].sample(n=min(SAMPLE_SIZE, len(train_A)), random_state=42)
train_A_sampled = train_A[train_A['Test_id'].isin(sampled_ids)].copy()
del train_A  # 메모리 해제

train_A_full = train_A_sampled.merge(train_meta[['Test_id', 'Label']], on='Test_id', how='left')
del train_A_sampled

print(f"A 검사 샘플: {train_A_full.shape}")
print(f"A 검사 위험군 비율: {train_A_full['Label'].mean():.4f}")

# B 검사
train_B = pd.read_csv("./data/train/B.csv")
print(f"B 검사 원본: {train_B.shape}")

sampled_ids_b = train_B['Test_id'].sample(n=min(SAMPLE_SIZE, len(train_B)), random_state=42)
train_B_sampled = train_B[train_B['Test_id'].isin(sampled_ids_b)].copy()
del train_B

train_B_full = train_B_sampled.merge(train_meta[['Test_id', 'Label']], on='Test_id', how='left')
del train_B_sampled

print(f"B 검사 샘플: {train_B_full.shape}")
print(f"B 검사 위험군 비율: {train_B_full['Label'].mean():.4f}")

print("\n데이터 로드 완료! (메모리 최적화됨)")

데이터 로드 중...


A 검사 원본: (647241, 37)
A 검사 샘플: (50000, 38)
A 검사 위험군 비율: 0.0228
B 검사 원본: (297526, 31)
B 검사 샘플: (50000, 32)
B 검사 위험군 비율: 0.0430

데이터 로드 완료! (메모리 최적화됨)


## 1. A1 검사 상세 분석: 행동반응 측정 (좌/우, 속도)

**검사 내용:**
- 18 trials
- Condition 1: 좌측(1) / 우측(2) 자극 (각 9회)
- Condition 2: 느린(1) / 보통(2) / 빠른(3) 속도 (각 6회)
- 반응 정확도와 반응 시간 측정

**분석 목표:**
- 좌/우 편향성 확인
- 속도별 반응 차이 확인
- 위험군의 특이 패턴 발견

In [25]:
# 헬퍼 함수: 시퀀스 데이터 파싱
def parse_sequence(series, dtype=str):
    """콤마로 구분된 시퀀스를 리스트로 변환"""
    def parse_row(x):
        if pd.isna(x) or x == '':
            return []
        try:
            if dtype == float:
                return [float(v) for v in str(x).split(',')]
            else:
                return str(x).split(',')
        except:
            return []
    return series.apply(parse_row)

def calculate_masked_mean(cond_series, val_series, mask_val):
    """특정 조건에서의 평균 계산"""
    results = []
    for cond, val in zip(cond_series, val_series):
        if len(cond) > 0 and len(val) > 0 and len(cond) == len(val):
            masked_vals = [v for c, v in zip(cond, val) if str(c) == str(mask_val)]
            results.append(np.mean(masked_vals) if len(masked_vals) > 0 else np.nan)
        else:
            results.append(np.nan)
    return pd.Series(results)

print("헬퍼 함수 정의 완료")

헬퍼 함수 정의 완료


In [26]:
# A1 검사 데이터 준비 (이미 샘플링된 데이터 사용)
df_a1 = train_A_full.copy()

print(f"A1 분석용 샘플 수: {len(df_a1)}")
print(f"위험군 비율: {df_a1['Label'].mean():.4f}")

A1 분석용 샘플 수: 50000
위험군 비율: 0.0228


In [27]:
# A1 피처 추출
print("A1 피처 추출 중...")

# 시퀀스 파싱
df_a1['A1_cond1_list'] = parse_sequence(df_a1['A1-1'], dtype=str)
df_a1['A1_cond2_list'] = parse_sequence(df_a1['A1-2'], dtype=str)
df_a1['A1_resp_list'] = parse_sequence(df_a1['A1-3'], dtype=str)
df_a1['A1_rt_list'] = parse_sequence(df_a1['A1-4'], dtype=float)

# 1. 전체 반응률 및 평균 반응시간
df_a1['A1_resp_rate'] = df_a1['A1_resp_list'].apply(
    lambda x: x.count('1') / len(x) if len(x) > 0 else np.nan
)
df_a1['A1_rt_mean'] = df_a1['A1_rt_list'].apply(
    lambda x: np.mean(x) if len(x) > 0 else np.nan
)
df_a1['A1_rt_std'] = df_a1['A1_rt_list'].apply(
    lambda x: np.std(x) if len(x) > 0 else np.nan
)

# 2. 좌/우 편향 분석
df_a1['A1_rt_left'] = calculate_masked_mean(
    df_a1['A1_cond1_list'], df_a1['A1_rt_list'], '1'
)
df_a1['A1_rt_right'] = calculate_masked_mean(
    df_a1['A1_cond1_list'], df_a1['A1_rt_list'], '2'
)
df_a1['A1_lr_diff'] = df_a1['A1_rt_left'] - df_a1['A1_rt_right']  # 양수면 왼쪽이 느림
df_a1['A1_lr_bias'] = df_a1['A1_lr_diff'].abs()  # 편향 크기

# 3. 속도별 반응 분석
df_a1['A1_rt_slow'] = calculate_masked_mean(
    df_a1['A1_cond2_list'], df_a1['A1_rt_list'], '1'
)
df_a1['A1_rt_normal'] = calculate_masked_mean(
    df_a1['A1_cond2_list'], df_a1['A1_rt_list'], '2'
)
df_a1['A1_rt_fast'] = calculate_masked_mean(
    df_a1['A1_cond2_list'], df_a1['A1_rt_list'], '3'
)
df_a1['A1_speed_adaptation'] = df_a1['A1_rt_slow'] - df_a1['A1_rt_fast']  # 속도 적응력

print("A1 피처 추출 완료!")
print("\n생성된 피처:")
a1_features = ['A1_resp_rate', 'A1_rt_mean', 'A1_rt_std', 'A1_lr_diff', 'A1_lr_bias', 'A1_speed_adaptation']
print(a1_features)

A1 피처 추출 중...
A1 피처 추출 완료!

생성된 피처:
['A1_resp_rate', 'A1_rt_mean', 'A1_rt_std', 'A1_lr_diff', 'A1_lr_bias', 'A1_speed_adaptation']


In [28]:
# A1: 좌/우 편향 분석
print("=" * 60)
print("A1 검사: 좌/우 편향 분석")
print("=" * 60)

for label in [0, 1]:
    label_name = "정상군" if label == 0 else "위험군"
    data = df_a1[df_a1['Label'] == label]
    
    print(f"\n[{label_name}]")
    print(f"평균 좌측 반응시간: {data['A1_rt_left'].mean():.2f}ms")
    print(f"평균 우측 반응시간: {data['A1_rt_right'].mean():.2f}ms")
    print(f"좌우 차이(좌-우): {data['A1_lr_diff'].mean():.2f}ms")
    print(f"편향 크기(절대값): {data['A1_lr_bias'].mean():.2f}ms")

# 통계 검정
risk_lr_bias = df_a1[df_a1['Label'] == 1]['A1_lr_bias'].dropna()
normal_lr_bias = df_a1[df_a1['Label'] == 0]['A1_lr_bias'].dropna()

if len(risk_lr_bias) > 0 and len(normal_lr_bias) > 0:
    t_stat, p_val = stats.ttest_ind(risk_lr_bias, normal_lr_bias, equal_var=False)
    print(f"\n[통계 검정] 좌우 편향 크기")
    print(f"t-statistic: {t_stat:.4f}, p-value: {p_val:.6f}")
    print("→ 유의함" if p_val < 0.05 else "→ 유의하지 않음")

A1 검사: 좌/우 편향 분석

[정상군]
평균 좌측 반응시간: -8.44ms
평균 우측 반응시간: -10.23ms
좌우 차이(좌-우): 1.79ms
편향 크기(절대값): 23.89ms

[위험군]
평균 좌측 반응시간: -11.93ms
평균 우측 반응시간: -13.16ms
좌우 차이(좌-우): 1.23ms
편향 크기(절대값): 26.27ms

[통계 검정] 좌우 편향 크기
t-statistic: 2.1921, p-value: 0.028570
→ 유의함


In [29]:
# 시각화: 좌/우 반응시간 비교
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 좌우 반응시간 산점도
for label, color, name in [(0, 'skyblue', 'Normal'), (1, 'salmon', 'Risk')]:
    data = df_a1[df_a1['Label'] == label]
    axes[0].scatter(data['A1_rt_left'], data['A1_rt_right'], 
                   alpha=0.3, s=20, color=color, label=name)

axes[0].plot([0, 1000], [0, 1000], 'k--', alpha=0.5, label='Equal')
axes[0].set_xlabel('Left Response Time (ms)', fontsize=12)
axes[0].set_ylabel('Right Response Time (ms)', fontsize=12)
axes[0].set_title('A1: Left vs Right Response Time', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# 좌우 편향 크기 분포
axes[1].hist(normal_lr_bias, bins=50, alpha=0.6, label='Normal', color='skyblue', density=True)
axes[1].hist(risk_lr_bias, bins=50, alpha=0.6, label='Risk', color='salmon', density=True)
axes[1].set_xlabel('L-R Bias (absolute ms)', fontsize=12)
axes[1].set_ylabel('Density', fontsize=12)
axes[1].set_title('A1: Left-Right Bias Distribution', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [30]:
# A1: 속도 적응력 분석
print("=" * 60)
print("A1 검사: 속도별 반응 분석")
print("=" * 60)

for label in [0, 1]:
    label_name = "정상군" if label == 0 else "위험군"
    data = df_a1[df_a1['Label'] == label]
    
    print(f"\n[{label_name}]")
    print(f"느린 자극 반응시간: {data['A1_rt_slow'].mean():.2f}ms")
    print(f"보통 자극 반응시간: {data['A1_rt_normal'].mean():.2f}ms")
    print(f"빠른 자극 반응시간: {data['A1_rt_fast'].mean():.2f}ms")
    print(f"속도 적응력(느림-빠름): {data['A1_speed_adaptation'].mean():.2f}ms")

# 통계 검정
risk_adapt = df_a1[df_a1['Label'] == 1]['A1_speed_adaptation'].dropna()
normal_adapt = df_a1[df_a1['Label'] == 0]['A1_speed_adaptation'].dropna()

if len(risk_adapt) > 0 and len(normal_adapt) > 0:
    t_stat, p_val = stats.ttest_ind(risk_adapt, normal_adapt, equal_var=False)
    print(f"\n[통계 검정] 속도 적응력")
    print(f"t-statistic: {t_stat:.4f}, p-value: {p_val:.6f}")
    print("→ 유의함" if p_val < 0.05 else "→ 유의하지 않음")

A1 검사: 속도별 반응 분석

[정상군]
느린 자극 반응시간: -8.04ms
보통 자극 반응시간: -5.19ms
빠른 자극 반응시간: -14.78ms
속도 적응력(느림-빠름): 6.74ms

[위험군]
느린 자극 반응시간: -12.41ms
보통 자극 반응시간: -6.88ms
빠른 자극 반응시간: -18.34ms
속도 적응력(느림-빠름): 5.93ms

[통계 검정] 속도 적응력
t-statistic: -0.4769, p-value: 0.633525
→ 유의하지 않음


In [31]:
# 시각화: 속도별 반응시간
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 그룹별 속도 반응 패턴
speed_data = []
for label, name in [(0, 'Normal'), (1, 'Risk')]:
    data = df_a1[df_a1['Label'] == label]
    speed_data.append({
        'Group': name,
        'Slow': data['A1_rt_slow'].mean(),
        'Normal': data['A1_rt_normal'].mean(),
        'Fast': data['A1_rt_fast'].mean()
    })

speed_df = pd.DataFrame(speed_data)
x = np.arange(3)
width = 0.35

axes[0].bar(x - width/2, [speed_df.loc[0, 'Slow'], speed_df.loc[0, 'Normal'], speed_df.loc[0, 'Fast']], 
           width, label='Normal', color='skyblue')
axes[0].bar(x + width/2, [speed_df.loc[1, 'Slow'], speed_df.loc[1, 'Normal'], speed_df.loc[1, 'Fast']], 
           width, label='Risk', color='salmon')
axes[0].set_xlabel('Stimulus Speed', fontsize=12)
axes[0].set_ylabel('Response Time (ms)', fontsize=12)
axes[0].set_title('A1: Response Time by Stimulus Speed', fontsize=14, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(['Slow', 'Normal', 'Fast'])
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# 속도 적응력 분포
axes[1].hist(normal_adapt, bins=50, alpha=0.6, label='Normal', color='skyblue', density=True)
axes[1].hist(risk_adapt, bins=50, alpha=0.6, label='Risk', color='salmon', density=True)
axes[1].set_xlabel('Speed Adaptation (Slow - Fast, ms)', fontsize=12)
axes[1].set_ylabel('Density', fontsize=12)
axes[1].set_title('A1: Speed Adaptation Distribution', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 2. A4 검사 상세 분석: Stroop 효과 측정

**검사 내용:**
- 80 trials (일치 40 / 불일치 40)
- Condition 1: 일치(1, congruent) / 불일치(2, incongruent)
- Condition 2: 빨강(1) / 초록(2)
- 정확도와 반응시간 측정

**Stroop 효과:** 글자 의미와 색상이 불일치할 때 반응시간이 증가하는 현상

**분석 목표:**
- Stroop 간섭 효과 크기 측정
- 위험군에서 간섭 효과가 더 큰지 확인
- 충동 억제 능력 평가

In [32]:
# A4 검사 데이터 준비 (같은 샘플 재사용)
df_a4 = train_A_full.copy()

print("A4 피처 추출 중...")

# 시퀀스 파싱
df_a4['A4_cond1_list'] = parse_sequence(df_a4['A4-1'], dtype=str)
df_a4['A4_cond2_list'] = parse_sequence(df_a4['A4-2'], dtype=str)
df_a4['A4_acc_list'] = parse_sequence(df_a4['A4-3'], dtype=str)
df_a4['A4_resp_list'] = parse_sequence(df_a4['A4-4'], dtype=str)
df_a4['A4_rt_list'] = parse_sequence(df_a4['A4-5'], dtype=float)

# 1. 전체 정확도 및 반응시간
df_a4['A4_acc_rate'] = df_a4['A4_acc_list'].apply(
    lambda x: x.count('1') / len(x) if len(x) > 0 else np.nan
)
df_a4['A4_rt_mean'] = df_a4['A4_rt_list'].apply(
    lambda x: np.mean(x) if len(x) > 0 else np.nan
)

# 2. Stroop 효과 분석
df_a4['A4_rt_congruent'] = calculate_masked_mean(
    df_a4['A4_cond1_list'], df_a4['A4_rt_list'], '1'
)
df_a4['A4_rt_incongruent'] = calculate_masked_mean(
    df_a4['A4_cond1_list'], df_a4['A4_rt_list'], '2'
)
df_a4['A4_stroop_effect'] = df_a4['A4_rt_incongruent'] - df_a4['A4_rt_congruent']  # Stroop 간섭

# 3. 색상별 반응시간
df_a4['A4_rt_red'] = calculate_masked_mean(
    df_a4['A4_cond2_list'], df_a4['A4_rt_list'], '1'
)
df_a4['A4_rt_green'] = calculate_masked_mean(
    df_a4['A4_cond2_list'], df_a4['A4_rt_list'], '2'
)

print("A4 피처 추출 완료!")

A4 피처 추출 중...
A4 피처 추출 완료!


In [33]:
# A4: Stroop 효과 분석
print("=" * 60)
print("A4 검사: Stroop 효과 분석")
print("=" * 60)

for label in [0, 1]:
    label_name = "정상군" if label == 0 else "위험군"
    data = df_a4[df_a4['Label'] == label]
    
    print(f"\n[{label_name}]")
    print(f"전체 정확도: {data['A4_acc_rate'].mean():.4f}")
    print(f"일치 조건 반응시간: {data['A4_rt_congruent'].mean():.2f}ms")
    print(f"불일치 조건 반응시간: {data['A4_rt_incongruent'].mean():.2f}ms")
    print(f"Stroop 간섭 효과: {data['A4_stroop_effect'].mean():.2f}ms")

# 통계 검정
risk_stroop = df_a4[df_a4['Label'] == 1]['A4_stroop_effect'].dropna()
normal_stroop = df_a4[df_a4['Label'] == 0]['A4_stroop_effect'].dropna()

if len(risk_stroop) > 0 and len(normal_stroop) > 0:
    t_stat, p_val = stats.ttest_ind(risk_stroop, normal_stroop, equal_var=False)
    print(f"\n[통계 검정] Stroop 간섭 효과")
    print(f"t-statistic: {t_stat:.4f}, p-value: {p_val:.6f}")
    print("→ 유의함" if p_val < 0.05 else "→ 유의하지 않음")
    
    if p_val < 0.05:
        effect_diff = risk_stroop.mean() - normal_stroop.mean()
        if effect_diff > 0:
            print(f"→ 위험군이 {effect_diff:.2f}ms 더 큰 간섭 효과 (충동 억제 능력 낮음)")
        else:
            print(f"→ 정상군이 {-effect_diff:.2f}ms 더 큰 간섭 효과")

A4 검사: Stroop 효과 분석

[정상군]
전체 정확도: 0.9864
일치 조건 반응시간: 624.99ms
불일치 조건 반응시간: 643.50ms
Stroop 간섭 효과: 18.50ms

[위험군]
전체 정확도: 0.9853
일치 조건 반응시간: 647.69ms
불일치 조건 반응시간: 668.98ms
Stroop 간섭 효과: 21.30ms

[통계 검정] Stroop 간섭 효과
t-statistic: 2.7328, p-value: 0.006373
→ 유의함
→ 위험군이 2.79ms 더 큰 간섭 효과 (충동 억제 능력 낮음)


In [34]:
# 시각화: Stroop 효과
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. 일치/불일치 조건 반응시간 비교
stroop_data = []
for label, name in [(0, 'Normal'), (1, 'Risk')]:
    data = df_a4[df_a4['Label'] == label]
    stroop_data.append({
        'Group': name,
        'Congruent': data['A4_rt_congruent'].mean(),
        'Incongruent': data['A4_rt_incongruent'].mean()
    })

stroop_df = pd.DataFrame(stroop_data)
x = np.arange(2)
width = 0.35

axes[0].bar(x - width/2, [stroop_df.loc[0, 'Congruent'], stroop_df.loc[0, 'Incongruent']], 
           width, label='Normal', color='skyblue')
axes[0].bar(x + width/2, [stroop_df.loc[1, 'Congruent'], stroop_df.loc[1, 'Incongruent']], 
           width, label='Risk', color='salmon')
axes[0].set_xlabel('Condition', fontsize=12)
axes[0].set_ylabel('Response Time (ms)', fontsize=12)
axes[0].set_title('A4: Stroop Effect (Congruent vs Incongruent)', fontsize=14, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(['Congruent', 'Incongruent'])
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# 2. Stroop 간섭 효과 분포
axes[1].hist(normal_stroop, bins=50, alpha=0.6, label='Normal', color='skyblue', density=True)
axes[1].hist(risk_stroop, bins=50, alpha=0.6, label='Risk', color='salmon', density=True)
axes[1].axvline(normal_stroop.mean(), color='blue', linestyle='--', linewidth=2, label='Normal Mean')
axes[1].axvline(risk_stroop.mean(), color='red', linestyle='--', linewidth=2, label='Risk Mean')
axes[1].set_xlabel('Stroop Interference (ms)', fontsize=12)
axes[1].set_ylabel('Density', fontsize=12)
axes[1].set_title('A4: Stroop Interference Distribution', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

# 3. 정확도 vs Stroop 효과 산점도
for label, color, name in [(0, 'skyblue', 'Normal'), (1, 'salmon', 'Risk')]:
    data = df_a4[df_a4['Label'] == label]
    axes[2].scatter(data['A4_acc_rate'], data['A4_stroop_effect'], 
                   alpha=0.3, s=20, color=color, label=name)

axes[2].set_xlabel('Accuracy Rate', fontsize=12)
axes[2].set_ylabel('Stroop Interference (ms)', fontsize=12)
axes[2].set_title('A4: Accuracy vs Stroop Effect', fontsize=14, fontweight='bold')
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 3. A5 검사 상세 분석: 변화 탐지 능력

**검사 내용:**
- 36 trials (변화없음 18 / 위치변화 6 / 색변화 6 / 모양변화 6)
- Condition: 1(변화없음) / 2(위치) / 3(색) / 4(모양)
- 정확도 측정

**분석 목표:**
- 변화 탐지 민감도 평가
- False Positive 비율 확인 (변화 없는데 있다고 답함)
- 주의력 및 작업 기억 능력 평가

In [35]:
# A5 검사 데이터 준비 (같은 샘플 재사용)
df_a5 = train_A_full.copy()

print("A5 피처 추출 중...")

# 시퀀스 파싱
df_a5['A5_cond_list'] = parse_sequence(df_a5['A5-1'], dtype=str)
df_a5['A5_acc_list'] = parse_sequence(df_a5['A5-2'], dtype=str)
df_a5['A5_resp_list'] = parse_sequence(df_a5['A5-3'], dtype=str)

# 1. 전체 정확도
df_a5['A5_acc_rate'] = df_a5['A5_acc_list'].apply(
    lambda x: x.count('1') / len(x) if len(x) > 0 else np.nan
)

# 2. 변화 없음 정확도 (True Negative)
def calc_nonchange_acc(cond_list, acc_list):
    if len(cond_list) == 0 or len(acc_list) == 0:
        return np.nan
    nonchange_accs = [int(a) for c, a in zip(cond_list, acc_list) if c == '1']
    return np.mean(nonchange_accs) if len(nonchange_accs) > 0 else np.nan

df_a5['A5_nonchange_acc'] = df_a5.apply(
    lambda row: calc_nonchange_acc(row['A5_cond_list'], row['A5_acc_list']), axis=1
)

# 3. 변화 있음 정확도 (True Positive)
def calc_change_acc(cond_list, acc_list):
    if len(cond_list) == 0 or len(acc_list) == 0:
        return np.nan
    change_accs = [int(a) for c, a in zip(cond_list, acc_list) if c in ['2', '3', '4']]
    return np.mean(change_accs) if len(change_accs) > 0 else np.nan

df_a5['A5_change_acc'] = df_a5.apply(
    lambda row: calc_change_acc(row['A5_cond_list'], row['A5_acc_list']), axis=1
)

# 4. 변화 탐지 민감도
df_a5['A5_sensitivity'] = df_a5['A5_change_acc'] - (1 - df_a5['A5_nonchange_acc'])  # d'

print("A5 피처 추출 완료!")

A5 피처 추출 중...
A5 피처 추출 완료!


In [36]:
# A5: 변화 탐지 능력 분석
print("=" * 60)
print("A5 검사: 변화 탐지 능력 분석")
print("=" * 60)

for label in [0, 1]:
    label_name = "정상군" if label == 0 else "위험군"
    data = df_a5[df_a5['Label'] == label]
    
    print(f"\n[{label_name}]")
    print(f"전체 정확도: {data['A5_acc_rate'].mean():.4f}")
    print(f"변화 없음 정확도 (Specificity): {data['A5_nonchange_acc'].mean():.4f}")
    print(f"변화 있음 정확도 (Sensitivity): {data['A5_change_acc'].mean():.4f}")
    print(f"False Positive 비율: {(1 - data['A5_nonchange_acc']).mean():.4f}")
    print(f"탐지 민감도 지표: {data['A5_sensitivity'].mean():.4f}")

# 통계 검정
risk_sens = df_a5[df_a5['Label'] == 1]['A5_sensitivity'].dropna()
normal_sens = df_a5[df_a5['Label'] == 0]['A5_sensitivity'].dropna()

if len(risk_sens) > 0 and len(normal_sens) > 0:
    t_stat, p_val = stats.ttest_ind(risk_sens, normal_sens, equal_var=False)
    print(f"\n[통계 검정] 변화 탐지 민감도")
    print(f"t-statistic: {t_stat:.4f}, p-value: {p_val:.6f}")
    print("→ 유의함" if p_val < 0.05 else "→ 유의하지 않음")

A5 검사: 변화 탐지 능력 분석

[정상군]
전체 정확도: 0.7838
변화 없음 정확도 (Specificity): 1.0941
변화 있음 정확도 (Sensitivity): 1.3382
False Positive 비율: -0.0941
탐지 민감도 지표: 1.4323



[위험군]
전체 정확도: 0.7651
변화 없음 정확도 (Specificity): 1.1061
변화 있음 정확도 (Sensitivity): 1.3636
False Positive 비율: -0.1061
탐지 민감도 지표: 1.4697

[통계 검정] 변화 탐지 민감도
t-statistic: 5.3195, p-value: 0.000000
→ 유의함


In [37]:
# 시각화: 변화 탐지 능력
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. 변화 없음 vs 변화 있음 정확도
change_data = []
for label, name in [(0, 'Normal'), (1, 'Risk')]:
    data = df_a5[df_a5['Label'] == label]
    change_data.append({
        'Group': name,
        'No Change': data['A5_nonchange_acc'].mean(),
        'Change': data['A5_change_acc'].mean()
    })

change_df = pd.DataFrame(change_data)
x = np.arange(2)
width = 0.35

axes[0, 0].bar(x - width/2, [change_df.loc[0, 'No Change'], change_df.loc[0, 'Change']], 
              width, label='Normal', color='skyblue')
axes[0, 0].bar(x + width/2, [change_df.loc[1, 'No Change'], change_df.loc[1, 'Change']], 
              width, label='Risk', color='salmon')
axes[0, 0].set_ylabel('Accuracy', fontsize=12)
axes[0, 0].set_title('A5: Change Detection Accuracy', fontsize=14, fontweight='bold')
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels(['No Change', 'Change'])
axes[0, 0].legend()
axes[0, 0].grid(axis='y', alpha=0.3)

# 2. 민감도 분포
axes[0, 1].hist(normal_sens, bins=50, alpha=0.6, label='Normal', color='skyblue', density=True)
axes[0, 1].hist(risk_sens, bins=50, alpha=0.6, label='Risk', color='salmon', density=True)
axes[0, 1].set_xlabel('Sensitivity', fontsize=12)
axes[0, 1].set_ylabel('Density', fontsize=12)
axes[0, 1].set_title('A5: Sensitivity Distribution', fontsize=14, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# 3. ROC-style plot (Sensitivity vs Specificity)
for label, color, name in [(0, 'skyblue', 'Normal'), (1, 'salmon', 'Risk')]:
    data = df_a5[df_a5['Label'] == label]
    axes[1, 0].scatter(1 - data['A5_nonchange_acc'], data['A5_change_acc'], 
                      alpha=0.3, s=20, color=color, label=name)

axes[1, 0].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random')
axes[1, 0].set_xlabel('False Positive Rate (1 - Specificity)', fontsize=12)
axes[1, 0].set_ylabel('True Positive Rate (Sensitivity)', fontsize=12)
axes[1, 0].set_title('A5: Sensitivity vs False Positive', fontsize=14, fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

# 4. False Positive 비율 박스플롯
fp_normal = (1 - df_a5[df_a5['Label'] == 0]['A5_nonchange_acc']).dropna()
fp_risk = (1 - df_a5[df_a5['Label'] == 1]['A5_nonchange_acc']).dropna()

axes[1, 1].boxplot([fp_normal, fp_risk], labels=['Normal', 'Risk'], patch_artist=True,
                   boxprops=dict(facecolor='lightblue'),
                   medianprops=dict(color='red', linewidth=2))
axes[1, 1].set_ylabel('False Positive Rate', fontsize=12)
axes[1, 1].set_title('A5: False Positive Rate Comparison', fontsize=14, fontweight='bold')
axes[1, 1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 4. B 검사 상세 분석

B 검사는 자격 유지 검사로, A 검사와 유사하지만 일부 다른 과제들이 포함됩니다.

**주요 검사:**
- B1, B2: 변화 탐지 (color change detection)
- B3: 기본 반응 검사
- B4: Stroop 유사 검사 (congruent/incongruent)
- B5: 기본 반응 검사
- B9, B10: 멀티태스킹 (청각 + 시각 동시 과제)

In [38]:
# B9/B10 검사: 멀티태스킹 능력 분석
print("=" * 60)
print("B9/B10 검사: 멀티태스킹 능력 분석")
print("=" * 60)

# B9 검사
b9_cols = ['B9-1', 'B9-2', 'B9-3', 'B9-4', 'B9-5']
if all(col in train_B_full.columns for col in b9_cols):
    print("\n[B9 검사 - 듀얼 태스크]")
    print("청각: target 15, distractor 35")
    print("시각: obstacle avoidance 32 trials")
    
    for label in [0, 1]:
        label_name = "정상군" if label == 0 else "위험군"
        data = train_B_full[train_B_full['Label'] == label]
        
        print(f"\n[{label_name}]")
        print(f"청각 Hit (평균): {data['B9-1'].mean():.2f}")
        print(f"청각 Miss (평균): {data['B9-2'].mean():.2f}")
        print(f"청각 False Alarm (평균): {data['B9-3'].mean():.2f}")
        print(f"청각 Correct Rejection (평균): {data['B9-4'].mean():.2f}")
        print(f"시각 Error (평균): {data['B9-5'].mean():.2f}")
        
        # 청각 정확도 계산
        aud_acc = (data['B9-1'] + data['B9-4']) / (data['B9-1'] + data['B9-2'] + data['B9-3'] + data['B9-4'])
        print(f"청각 전체 정확도: {aud_acc.mean():.4f}")
else:
    print("B9 검사 데이터가 없습니다.")

# B10 검사
b10_cols = ['B10-1', 'B10-2', 'B10-3', 'B10-4', 'B10-5', 'B10-6']
if all(col in train_B_full.columns for col in b10_cols):
    print("\n[B10 검사 - 트리플 태스크]")
    print("청각: target 20, distractor 60")
    print("시각1: obstacle avoidance 52 trials")
    print("시각2: color selection 20 trials")
    
    for label in [0, 1]:
        label_name = "정상군" if label == 0 else "위험군"
        data = train_B_full[train_B_full['Label'] == label]
        
        print(f"\n[{label_name}]")
        print(f"청각 Hit (평균): {data['B10-1'].mean():.2f}")
        print(f"청각 Miss (평균): {data['B10-2'].mean():.2f}")
        print(f"청각 False Alarm (평균): {data['B10-3'].mean():.2f}")
        print(f"청각 Correct Rejection (평균): {data['B10-4'].mean():.2f}")
        print(f"시각1 Error (평균): {data['B10-5'].mean():.2f}")
        print(f"시각2 정답 (평균): {data['B10-6'].mean():.2f}")
        
        # 전체 정확도
        aud_acc = (data['B10-1'] + data['B10-4']) / (data['B10-1'] + data['B10-2'] + data['B10-3'] + data['B10-4'])
        vis2_acc = data['B10-6'] / 20
        print(f"청각 전체 정확도: {aud_acc.mean():.4f}")
        print(f"시각2 정확도: {vis2_acc.mean():.4f}")
else:
    print("B10 검사 데이터가 없습니다.")

B9/B10 검사: 멀티태스킹 능력 분석

[B9 검사 - 듀얼 태스크]
청각: target 15, distractor 35
시각: obstacle avoidance 32 trials

[정상군]
청각 Hit (평균): 13.79
청각 Miss (평균): 1.21
청각 False Alarm (평균): 0.84
청각 Correct Rejection (평균): 34.15
시각 Error (평균): 2.28
청각 전체 정확도: 0.9589

[위험군]
청각 Hit (평균): 13.50
청각 Miss (평균): 1.50
청각 False Alarm (평균): 1.00
청각 Correct Rejection (평균): 34.00
시각 Error (평균): 2.48
청각 전체 정확도: 0.9500

[B10 검사 - 트리플 태스크]
청각: target 20, distractor 60
시각1: obstacle avoidance 52 trials
시각2: color selection 20 trials

[정상군]
청각 Hit (평균): 17.28
청각 Miss (평균): 2.72
청각 False Alarm (평균): 2.55
청각 Correct Rejection (평균): 57.45
시각1 Error (평균): 4.74
시각2 정답 (평균): 18.59
청각 전체 정확도: 0.9341
시각2 정확도: 0.9297

[위험군]
청각 Hit (평균): 16.86
청각 Miss (평균): 3.14
청각 False Alarm (평균): 2.93
청각 Correct Rejection (평균): 57.07
시각1 Error (평균): 5.18
시각2 정답 (평균): 18.70
청각 전체 정확도: 0.9241
시각2 정확도: 0.9352


## 5. 종합 인지 프로필 분석

여러 검사 결과를 종합하여 운수종사자의 인지 프로필을 구성합니다.

In [39]:
# 종합 피처 생성
print("종합 인지 프로필 생성 중...")

# A 검사 주요 피처 병합
cognitive_profile = df_a1[['Test_id', 'Label', 'A1_resp_rate', 'A1_rt_mean', 'A1_lr_bias', 'A1_speed_adaptation']].copy()
cognitive_profile = cognitive_profile.merge(
    df_a4[['Test_id', 'A4_acc_rate', 'A4_stroop_effect']], on='Test_id', how='left'
)
cognitive_profile = cognitive_profile.merge(
    df_a5[['Test_id', 'A5_acc_rate', 'A5_sensitivity']], on='Test_id', how='left'
)

print(f"종합 프로필 생성 완료: {cognitive_profile.shape}")
print("\n주요 피처:")
print(cognitive_profile.columns.tolist())

종합 인지 프로필 생성 중...
종합 프로필 생성 완료: (50000, 10)

주요 피처:
['Test_id', 'Label', 'A1_resp_rate', 'A1_rt_mean', 'A1_lr_bias', 'A1_speed_adaptation', 'A4_acc_rate', 'A4_stroop_effect', 'A5_acc_rate', 'A5_sensitivity']


In [40]:
# 피처 간 상관관계 분석
feature_cols = ['A1_resp_rate', 'A1_rt_mean', 'A1_lr_bias', 'A1_speed_adaptation',
                'A4_acc_rate', 'A4_stroop_effect', 'A5_acc_rate', 'A5_sensitivity', 'Label']

corr_matrix = cognitive_profile[feature_cols].corr()

print("[Label과의 상관관계]")
label_corr = corr_matrix['Label'].sort_values(ascending=False)
print(label_corr)

# 시각화
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='RdBu_r', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Cognitive Profile - Feature Correlation Matrix', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

[Label과의 상관관계]
Label                  1.000000
A5_sensitivity         0.025635
A4_stroop_effect       0.012875
A1_lr_bias             0.010517
A1_resp_rate           0.007904
A1_speed_adaptation   -0.002075
A1_rt_mean            -0.007377
A4_acc_rate           -0.007802
A5_acc_rate           -0.025619
Name: Label, dtype: float64


In [41]:
# 레이더 차트: 정상군 vs 위험군 인지 프로필
from math import pi

# 정규화된 점수 계산 (0~1 범위)
radar_features = ['A1_resp_rate', 'A4_acc_rate', 'A5_acc_rate']  # 높을수록 좋음
radar_features_inv = ['A1_rt_mean', 'A1_lr_bias', 'A4_stroop_effect']  # 낮을수록 좋음

def normalize_score(series, inverse=False):
    """0~1 범위로 정규화, inverse=True면 낮을수록 좋은 것"""
    min_val, max_val = series.min(), series.max()
    normalized = (series - min_val) / (max_val - min_val) if max_val > min_val else series * 0
    return 1 - normalized if inverse else normalized

radar_data = []
for label, name in [(0, 'Normal'), (1, 'Risk')]:
    data = cognitive_profile[cognitive_profile['Label'] == label]
    profile = {
        'Group': name,
        'Response Accuracy': data['A1_resp_rate'].mean(),
        'Response Speed': 1000 / data['A1_rt_mean'].mean(),  # 역수로 빠를수록 높음
        'Motor Balance': 1 / (1 + data['A1_lr_bias'].mean()),  # 편향 작을수록 높음
        'Inhibition Control': 1 / (1 + data['A4_stroop_effect'].mean()),  # 간섭 작을수록 높음
        'Attention Accuracy': data['A5_acc_rate'].mean(),
        'Change Detection': (data['A5_sensitivity'].mean() + 1) / 2  # -1~1 → 0~1
    }
    radar_data.append(profile)

radar_df = pd.DataFrame(radar_data)
categories = list(radar_df.columns[1:])
N = len(categories)

# 각도 계산
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]

# 플롯
fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(projection='polar'))

for idx, row in radar_df.iterrows():
    values = row[1:].values.tolist()
    values += values[:1]
    
    color = 'skyblue' if row['Group'] == 'Normal' else 'salmon'
    ax.plot(angles, values, 'o-', linewidth=2, label=row['Group'], color=color)
    ax.fill(angles, values, alpha=0.25, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, size=11)
ax.set_ylim(0, 1)
ax.set_title('Cognitive Profile: Normal vs Risk', size=16, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
ax.grid(True)

plt.tight_layout()
plt.show()

print("\n[인지 프로필 점수]")
print(radar_df)


[인지 프로필 점수]
    Group  Response Accuracy  Response Speed  Motor Balance  \
0  Normal           0.014631     -107.090555       0.040179   
1    Risk           0.018242      -79.720345       0.036666   

   Inhibition Control  Attention Accuracy  Change Detection  
0            0.051271            0.783815          1.216157  
1            0.044849            0.765145          1.234834  


## 6. 핵심 인사이트 및 피처 엔지니어링 제안

In [42]:
print("=" * 80)
print("심층 분석 핵심 인사이트")
print("=" * 80)

insights = """
1. A1 검사 (행동반응 - 좌우/속도)
   ✓ 좌우 편향 크기가 위험 예측에 중요할 수 있음
   ✓ 속도 적응력 (느림-빠름 차이)이 주의력 유연성 반영
   
   권장 피처:
   - A1_lr_bias (좌우 편향 절대값)
   - A1_speed_adaptation (속도별 반응시간 차이)
   - A1_rt_cv (반응시간 변동계수 = std/mean)

2. A4 검사 (Stroop 효과)
   ✓ Stroop 간섭 효과 크기가 충동 억제 능력 반영
   ✓ 위험군에서 간섭 효과가 더 클 가능성
   
   권장 피처:
   - A4_stroop_effect (불일치-일치 반응시간)
   - A4_stroop_acc_drop (일치 vs 불일치 정확도 차이)
   - A4_speed_accuracy_tradeoff (반응시간/정확도)

3. A5 검사 (변화 탐지)
   ✓ False Positive 비율이 충동성/주의력 반영
   ✓ Sensitivity가 작업 기억 능력 반영
   
   권장 피처:
   - A5_sensitivity (변화 탐지 민감도)
   - A5_false_positive_rate (1 - nonchange_acc)
   - A5_response_bias (liberal vs conservative)

4. B9/B10 검사 (멀티태스킹)
   ✓ 멀티태스킹 정확도가 실제 운전 상황과 유사
   ✓ 청각+시각 동시 처리 능력이 중요
   
   권장 피처:
   - B9/B10 청각 정확도
   - B9/B10 시각 에러율
   - 듀얼/트리플 태스크 종합 점수

5. 복합 피처 (Cross-task features)
   ✓ 여러 검사의 일관성 체크
   ✓ 인지 프로필 종합 점수
   
   권장 피처:
   - 전체 반응시간 평균/표준편차
   - 전체 정확도 평균
   - 검사 간 일관성 지표 (반응시간 상관관계 등)
   - 인지 부하 저항성 (복잡한 과제 vs 단순 과제 차이)

6. 시간적 특성 (이력이 있는 경우)
   ✓ 동일 운수종사자의 이전 검사와 비교
   ✓ 인지 기능 변화 추세
   
   권장 피처:
   - 이전 검사 대비 변화율
   - 검사 간격
   - 악화/개선 추세

7. 모델링 전략
   ✓ A/B 검사 별도 모델 유지
   ✓ 각 검사별 특화 피처 활용
   ✓ 트리 기반 모델 + 신경망 앙상블 고려
   ✓ 클래스 가중치 조정 필수
"""

print(insights)
print("\n분석 완료! 🚗")

심층 분석 핵심 인사이트

1. A1 검사 (행동반응 - 좌우/속도)
   ✓ 좌우 편향 크기가 위험 예측에 중요할 수 있음
   ✓ 속도 적응력 (느림-빠름 차이)이 주의력 유연성 반영

   권장 피처:
   - A1_lr_bias (좌우 편향 절대값)
   - A1_speed_adaptation (속도별 반응시간 차이)
   - A1_rt_cv (반응시간 변동계수 = std/mean)

2. A4 검사 (Stroop 효과)
   ✓ Stroop 간섭 효과 크기가 충동 억제 능력 반영
   ✓ 위험군에서 간섭 효과가 더 클 가능성

   권장 피처:
   - A4_stroop_effect (불일치-일치 반응시간)
   - A4_stroop_acc_drop (일치 vs 불일치 정확도 차이)
   - A4_speed_accuracy_tradeoff (반응시간/정확도)

3. A5 검사 (변화 탐지)
   ✓ False Positive 비율이 충동성/주의력 반영
   ✓ Sensitivity가 작업 기억 능력 반영

   권장 피처:
   - A5_sensitivity (변화 탐지 민감도)
   - A5_false_positive_rate (1 - nonchange_acc)
   - A5_response_bias (liberal vs conservative)

4. B9/B10 검사 (멀티태스킹)
   ✓ 멀티태스킹 정확도가 실제 운전 상황과 유사
   ✓ 청각+시각 동시 처리 능력이 중요

   권장 피처:
   - B9/B10 청각 정확도
   - B9/B10 시각 에러율
   - 듀얼/트리플 태스크 종합 점수

5. 복합 피처 (Cross-task features)
   ✓ 여러 검사의 일관성 체크
   ✓ 인지 프로필 종합 점수

   권장 피처:
   - 전체 반응시간 평균/표준편차
   - 전체 정확도 평균
   - 검사 간 일관성 지표 (반응시간 상관관계 등)
   - 인지 부하 저항성 (복잡한 과제 vs 단순 과제 차이)

6. 시간적 특성 (이력